In [1]:
# 从这里开始我们思考如何实现loss函数部分

In [2]:
import torch 

from pytorch3d.io import load_objs_as_meshes
from pytorch3d.structures import Meshes
import drone_renderer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

render = drone_renderer.DroneRenderer(mesh_path= "data/sample/sample.obj",device= device)
mesh = render.mesh

B = 4  # 批量大小
P = torch.rand(B, 3,device=device) *10.0  # 假设的无人机中心点坐标



In [3]:
from pytorch3d.ops import sample_points_from_meshes, knn_points

# 1. 将 Mesh 转换为点云 (Point Cloud) 以供 KNN 使用
# 我们直接使用 render 对象中已经加载好的 mesh
# num_samples 决定了障碍物点云的密度，点越多计算越精确但开销越大
num_samples = 20000 
obstacle_pcd = sample_points_from_meshes(render.mesh, num_samples=num_samples)

print(f"生成的障碍物点云形状: {obstacle_pcd.shape}")  # 预期: (1, 20000, 3)

生成的障碍物点云形状: torch.Size([1, 20000, 3])


In [4]:
p1 = P.unsqueeze(1)  # 形状变为 (B, 1, 3)
p2 = obstacle_pcd  # 形状为 (1, N, 3)，N 是点云中的点数
p2 = p2.expand(B,-1,-1)  # 扩展为 (B, N, 3) 以匹配无人机批量大小
print(f"扩展后的障碍物点云形状: {p2.shape}")  # 预期: (B, 20000, 3)

dists = knn_points(p1, p2, K=1)
print(f"计算得到的最近距离形状: {dists.dists.shape}")  # 预期: (B, 1, 1)
print("距离结果",dists)
print("纯距离",dists.dists)  # 打印距离值以检查
print("形状",dists.dists.shape)
dists.dists.squeeze(-1)
print("去掉最后一个维度后的形状",dists.dists.squeeze(-1).shape)



扩展后的障碍物点云形状: torch.Size([4, 20000, 3])
计算得到的最近距离形状: torch.Size([4, 1, 1])
距离结果 KNN(dists=tensor([[[11.9785]],

        [[31.5639]],

        [[ 5.6733]],

        [[54.1752]]], device='cuda:0'), idx=tensor([[[16948]],

        [[ 1098]],

        [[ 4376]],

        [[ 4890]]], device='cuda:0'), knn=None)
纯距离 tensor([[[11.9785]],

        [[31.5639]],

        [[ 5.6733]],

        [[54.1752]]], device='cuda:0')
形状 torch.Size([4, 1, 1])
去掉最后一个维度后的形状 torch.Size([4, 1])


In [5]:

def calc_min_distance(drone_pos, obstacle_pcd):
    """
    计算无人机中心点与障碍物点云之间的最短距离 (Single Step)
    """
    p1 = drone_pos.unsqueeze(1) 
    B = p1.shape[0]
    
    if obstacle_pcd.shape[0] != B:
        obstacle_pcd_expanded = obstacle_pcd.expand(B, -1, -1)
    else:
        obstacle_pcd_expanded = obstacle_pcd
        
    result = knn_points(p1, obstacle_pcd_expanded, K=1)
    sq_dists = result.dists.squeeze(-1) # (B, 1) -> (B,)
    dists = torch.sqrt(sq_dists + 1e-6).squeeze(-1) 
    return dists

def calc_interpolated_distances(p_start, p_end, obstacle_pcd, steps=10):
    """
    计算从 p_start 到 p_end 路径上的插值点到障碍物的距离 (Sub-stepping)
    解决隧穿效应 (Tunneling Effect) 以及提供更平滑的碰撞梯度。
    
    Args:
        p_start, p_end: (B, 3) 起点和终点
        obstacle_pcd: (1, N, 3) or (B, N, 3) 障碍物点云
        steps: 插值步数 (参考项目使用了 10)
        
    Returns:
        dists: (B, steps) 每个插值点的欧氏距离
    """
    B = p_start.shape[0]
    device = p_start.device
    
    # 1. 线性插值生成路径点
    # alphas: (1, steps, 1) 范围 [0, 1]
    alphas = torch.linspace(0, 1, steps, device=device).view(1, steps, 1)
    
    # 插值公式: p(alpha) = p1 + (p2 - p1) * alpha
    # (B, 1, 3) + (B, 1, 3) * (1, steps, 1) -> (B, steps, 3)
    traj_points = p_start.unsqueeze(1) + (p_end - p_start).unsqueeze(1) * alphas
    
    # 2. 准备 KNN 输入
    # obstacle_pcd 扩展: (B, N, 3)
    if obstacle_pcd.shape[0] != B:
        obstacle_input = obstacle_pcd.expand(B, -1, -1)
    else:
        obstacle_input = obstacle_pcd
        
    # 3. 计算距离 
    # knn_points 支持输入: P1=(B, Steps, 3), P2=(B, N_obs, 3)
    # output.dists: (B, Steps, K) - 平方距离
    knn_res = knn_points(traj_points, obstacle_input, K=1)
    
    # 开根号得到欧氏距离
    dists = torch.sqrt(knn_res.dists.squeeze(-1) + 1e-6) # (B, steps)
    return dists

# 测试一下 单帧 距离计算
min_distances = calc_min_distance(P, obstacle_pcd)
print("部分无人机到障碍物的最短距离 (Single Step):", min_distances)

部分无人机到障碍物的最短距离 (Single Step): tensor([3.4610, 5.6182, 2.3819, 7.3604], device='cuda:0')


In [ ]:
import torch.nn.functional as F

class DroneLoss:
    """
    损失函数计算类，参考 DiffPhysDrone 项目实现。
    
    Code Review Notes:
    1. Collision Loss: 采用了线性插值 (Linear Interpolation) 替代参考项目的速度外推 (Velocity Extrapolation)，
       这在已知 next_state 的情况下更准确。
    2. Snap Loss: 参考项目计算的是推力方向 (Angular) 的 Snap，本实现计算的是线加速度 (Linear) 的 Snap。
       鉴于参考项目中该项权重默认为 0.0，此差异可忽略。
    """
    def __init__(self, device, margin=0.2, dt=1/15.0):
        self.device = device
        self.margin = margin # 安全边际 (radius + safety buffer) (参考项目 env.margin)
        self.dt = dt
        
        # 权重系数 (DiffPhysDrone/main_cuda.py args)
        self.coef_collide = 2.0
        self.coef_obj_avoidance = 1.5
        self.coef_v = 1.0
        self.coef_d_acc = 0.01
        self.coef_d_jerk = 0.001
        self.coef_d_snap = 0.00 # Reference default is 0.0
        
        # Magic Numbers from Reference
        self.V_TO_PT_SCALE = 135.0
        self.COLLISION_SCALE = -32.0
        
    def barrier(self, x, v_to_pt):
        """
        障碍物避障 Barrier Loss
        x: 实际距离 - margin
        v_to_pt: 接近速度权重 (由 diff(distance) 计算)
        Formula: mean( v * ReLU(1 - x)^2 )
        Reference: main_cuda.py barrier()
        """
        # 注意: x 是已经减去 margin 后的距离，如果 x < 0 说明小于 margin (潜在碰撞)
        # reference: (1-x).relu() 当 x 很小时 (比如 -0.1)，value > 1，惩罚大
        # 当 x (distance) 很大时，(1-x) 为负，relu 为 0，无损失。
        # 这里的 1.0 实际上也是一个隐含的 threshold 距离 (margin + 1.0)
        return (v_to_pt * (1 - x).relu().pow(2)).mean()

    def get_collision_loss(self, p_start, p_end, obstacle_pcd, sub_steps=10):
        """
        计算碰撞和避障损失
        Args:
            p_start, p_end: (B, 3) 无人机当前步和下一步的位置
            obstacle_pcd: (1, N, 3) 障碍物点云
        """
        # 1. 计算插值路径上的距离 (Calls global calc_interpolated_distances)
        # Review: 参考项目使用 p + v * sub_div (基于速度的外推)。
        # 这里使用 p_start 到 p_end 的线性插值。如果 p_end 是物理更新后的位置，这种“弦”插值也是合理的。
        dists = calc_interpolated_distances(p_start, p_end, obstacle_pcd, steps=sub_steps)
        
        # 2. 减去安全边际
        eff_dists = dists - self.margin
        
        # 3. 计算“接近速度”
        # Time derivative estimation across sub-steps
        # Reference: v_to_pt = (-diff * 135).clamp_min(1)
        # diff 的维度是 dim=1 (sub_steps 维度)，表示在该微小时间段内距离的变化率
        diff = torch.diff(eff_dists, dim=1)
        v_to_pt = (-diff * self.V_TO_PT_SCALE).clamp_min(1.0)
        
        # 对应 diff 后的距离数组 (减少了一个长度)
        curr_dist = eff_dists[:, 1:]
        
        # Collide Loss: 使用 Softplus 实现平滑的指数惩罚
        loss_collide = F.softplus(curr_dist * self.COLLISION_SCALE).mul(v_to_pt).mean()

        # Avoidance Loss: 使用 Barrier 函数推离障碍物
        loss_obj = self.barrier(curr_dist, v_to_pt)
        
        total = self.coef_collide * loss_collide + self.coef_obj_avoidance * loss_obj
        return total, loss_collide, loss_obj

    def get_smoothness_loss(self, acc_seq):
        """
        计算动作平滑度损失 (Acc, Jerk, Snap)
        Args:
            acc_seq: (B, T, 3) 加速度序列 (或 Force/Thrust)
        """
        # 确保输入有时间维度
        if acc_seq.dim() == 2:
             acc_seq = acc_seq.unsqueeze(1) # (B, 1, 3)
             
        T = acc_seq.shape[1]
        
        loss_acc = acc_seq.pow(2).sum(-1).mean()
        
        loss_jerk = torch.tensor(0.0, device=self.device)
        loss_snap = torch.tensor(0.0, device=self.device)
        
        if T >= 2:
            scale_fps = 1.0 / self.dt
            # Jerk: accelerations derivative
            jerk = torch.diff(acc_seq, dim=1) * scale_fps
            loss_jerk = jerk.pow(2).sum(-1).mean()
            
            if T >= 3:
                # Snap: jerk derivative
                # NOTE: Reference implementation calculates Snap on "Normalized Thrust Direction"
                # (removing gravity), penalizing angular angular acceleration.
                # Current implementation penalizes linear snap.
                snap = torch.diff(jerk, dim=1) * scale_fps
                loss_snap = snap.pow(2).sum(-1).mean()
        
        total = self.coef_d_acc * loss_acc + \
                self.coef_d_jerk * loss_jerk + \
                self.coef_d_snap * loss_snap
                
        return total, loss_acc, loss_jerk, loss_snap

    def get_velocity_loss(self, current_v, target_v, use_sliding_window=True, window_size=30):
        """
        计算速度追踪损失。
        Args:
            current_v: (B, T, 3) 历史速度
            target_v: (B, T, 3) 历史期望速度
        """
        if current_v.dim() == 2:
             current_v = current_v.unsqueeze(1)
             target_v = target_v.unsqueeze(1)
             
        T = current_v.shape[1]

        # Review: 严格匹配 main_cuda.py 中的 sliding window 逻辑
        if use_sliding_window and T > window_size:
            v_cum = current_v.cumsum(dim=1)
            # 计算滑动窗口平均速度
            v_avg = (v_cum[:, window_size:] - v_cum[:, :-window_size]) / window_size
            
            # 对齐 Target: reference slices target [1 : -29] (length T-30)
            # v_avg length is T-30.
            # 这里的切片逻辑需确保长度对齐:
            tgt_slice = target_v[:, 1:-window_size+1] 
            
            min_len = min(v_avg.shape[1], tgt_slice.shape[1])
            v_avg = v_avg[:, :min_len]
            tgt_slice = tgt_slice[:, :min_len]
            
            delta_v = torch.norm(v_avg - tgt_slice, p=2, dim=-1)
            loss_v = F.smooth_l1_loss(delta_v, torch.zeros_like(delta_v))
        else:
            loss_v = F.smooth_l1_loss(current_v, target_v)
            
        return self.coef_v * loss_v

# 初始化 Loss 计算器
# 注意: 这里的 dt 应该与你的模拟器设置保持一致 (0.02s)
drone_loss = DroneLoss(device=device, dt=0.02)

print("DroneLoss 模块加载成功，参数已初始化: dt=0.02")

DroneLoss 模块加载成功，参数已初始化: dt=0.02


In [9]:
# --- 构造 [碰撞测试] 用例 ---

# 1. 强制将无人机移动到障碍物的一个点上 (制造碰撞)
# 取障碍物点云的第一个点作为目标位置
target_obstacle_point = obstacle_pcd[0, 0] # (3,)
print(f"障碍物某点坐标: {target_obstacle_point.cpu().numpy()}")

# 设置 P_collision 为刚好在这个点附近 (距离 0.05m，小于 margin 0.2m)
# 这样应该会触发强烈的 Collision Loss
P_collision = target_obstacle_point + torch.tensor([0.05, 0.0, 0.0], device=device)
P_collision = P_collision.unsqueeze(0).expand(B, -1) # 扩展到 Batch

# P_next 继续往里撞
P_next_collision = P_collision + torch.tensor([0.1, 0.0, 0.0], device=device)

# 计算 Loss
loss_val_c, l_col_c, l_obj_c = drone_loss.get_collision_loss(P_collision, P_next_collision, obstacle_pcd)

print("\n--- 碰撞测试结果 (距离 < Margin) ---")
print(f"无人机位置: {P_collision[0].cpu().numpy()}")
print(f"Total Loss: {loss_val_c.item():.4f}")
print(f"  - Collide (软碰撞): {l_col_c.item():.4f} (应该很大)")
print(f"  - Avoid (避障场):   {l_obj_c.item():.4f} (应该有值)")


# --- 构造 [避障测试] 用例 ---
# 距离 0.5m ( 大于 margin 0.2, 小于 1.2)
# 应该只有 Obj Avoid Loss，没有 Collide Loss (或很小)
P_avoid = target_obstacle_point + torch.tensor([0.5, 0.0, 0.0], device=device)
P_avoid = P_avoid.unsqueeze(0).expand(B, -1)
P_next_avoid = P_avoid + torch.tensor([0.1, 0.0, 0.0], device=device)

loss_val_a, l_col_a, l_obj_a = drone_loss.get_collision_loss(P_avoid, P_next_avoid, obstacle_pcd)

print("\n--- 避障测试结果 (Margin < 距离 < 1.0) ---")
print(f"Total Loss: {loss_val_a.item():.4f}")
print(f"  - Collide (软碰撞): {l_col_a.item():.4f} (应该接近0)")
print(f"  - Avoid (避障场):   {l_obj_a.item():.4f} (应该有值)")

障碍物某点坐标: [ 82.0831   -1.      -97.69229]

--- 碰撞测试结果 (距离 < Margin) ---
无人机位置: [ 82.1331   -1.      -97.69229]
Total Loss: 7.9777
  - Collide (软碰撞): 3.0899 (应该很大)
  - Avoid (避障场):   1.1986 (应该有值)

--- 避障测试结果 (Margin < 距离 < 1.0) ---
Total Loss: 0.6242
  - Collide (软碰撞): 0.0000 (应该接近0)
  - Avoid (避障场):   0.4161 (应该有值)
